# Generate GLOBAL DBOF (`generate-global`)

A single CLI command (`generate-global`) and a single config file
(`configs/global/run/run.yaml`) replace the three legacy scripts
(`generate-global`, `generate-global --pipeline OSN`, `generate-global --pipeline DEPTH`).

The **`--pipeline`** argument selects the data source; **`--subset`** (or
`active_subsets` in the YAML) selects which property group to compute.

### Pipeline variants

| `--pipeline` | Data source | Notes |
|---|---|---|
| `SURF` | OSN kerchunk + S3 forcing | Core ocean vars from kerchunk; wind/ice from S3 |
| `OSN` | OSN kerchunk only | All surface + wind vars from kerchunk endpoints |
| `DEPTH` | S3 timestep stores (full depth) | 3D fields reduced to 2D via depth strategies |

### Surface subsets (SURF / OSN)

| `--subset` | Fields | Output zarr |
|---|---|---|
| `native_fields` | `Theta`, `Salt`, `Eta`, `W`, `U`, `V` (U/V rotated to eastward/northward) | `native_fields.zarr` |
| `surface_wind` | `oceTAUX`, `oceTAUY` (eastward/northward), `wind_stress_curl`, `ekman_pumping`, `u/v_ekman` (+ `oceQnet`, SURF only) | `surface_wind.zarr` |
| `icearea` | `SIarea` | `icearea.zarr` |
| `frontal_structure` | `gradb2`, `gradsalt2`, `gradtheta2`, `gradeta2`, `gradrho2`, `turner_angle`, `density`, `buoyancy` | `frontal_structure.zarr` |
| `kinematic` | `relative_vorticity`, `strain_n/s`, `strain_mag`, `divergence`, `coriolis_f`, `rossby_number`, `okubo_weiss` | `kinematic.zarr` |
| `frontogenesis` | `frontogenesis_tendency`, `ug`, `vg`, `frontogenesis_geo`, `frontogenesis_ageo` | `frontogenesis.zarr` |

### Depth subsets (DEPTH)

| `--subset` | Fields (expanded with depth suffixes) | Output zarr |
|---|---|---|
| `stratification` | `N2` x depths + `mixed_layer_depth`, `ml_heat_content` | `stratification.zarr` |
| `vertical_shear` | `vertical_shear`, `Ri` x depths | `vertical_shear.zarr` |
| `mixing_parameters` | `Fr`, `Ro`, `Bu` x depths | `mixing_parameters.zarr` |
| `ertel_pv` | `ertel_pv`, `ertel_pv_vertical`, `ertel_pv_tilt` x depths | `ertel_pv.zarr` |
| `buoyancy_fluxes` | `uB`, `vB`, `wB` x depths | `buoyancy_fluxes.zarr` |
| `surface_wind` | `oceTAUX`, `oceTAUY`, `oceQnet`, `wind_stress_curl`, `ekman_pumping`, `u/v_ekman` | `surface_wind.zarr` |
| `energetics` | `KE` x depths | `energetics.zarr` |
| `frontal_structure` | gradient fields x depths | `frontal_structure.zarr` |
| `kinematic` | vorticity/strain fields x depths | `kinematic.zarr` |
| `frontogenesis` | frontogenesis fields x depths (`ug/vg_sfc`) | `frontogenesis.zarr` |
| `native_fields` | `Theta`, `Salt`, `U`, `V`, `W` x depths, `Eta_sfc` (U/V eastward/northward) | `native_fields.zarr` |
| `icearea` | `SIarea` | `icearea.zarr` |

Depth suffixes: `_sfc` (surface), `_z25m` (25 m), `_mld` (mixed-layer depth), `_mld_mean` (ML mean).

---

### Output layout

All subsets share the same S3 directory via a shared `run_id`:
```
s3://{bucket}/{folder}/{run_id}/{date_prefix}/native_fields.zarr
s3://{bucket}/{folder}/{run_id}/{date_prefix}/kinematic.zarr
...
```

In [1]:
import datetime

# Initial set up

In the future we will want to update this to use docker or conda for end users.

## Local machine setup

#### Build the project
- `pip install .`

The `generate-global` entry point is registered in `pyproject.toml`:
```toml
generate-global = "dbof.cli.generate_global:main"
```

#### Install aws cli (optional)
You will want to install this if you want to manually see the data stored in the s3 bucket.

Example for installing on linux:
- `sudo curl "https://awscli.amazonaws.com/awscli-exe-linux-x86_64.zip" -o "awscliv2.zip"`
- `sudo unzip awscliv2.zip`
- `sudo ./aws/install`

## Running in NRP Jupyterhub

This is for running this notebook on NRP Jupyterhub.

This block simply builds the project and installs dependencies not already present on NRP Jupyterhub.
You can safely ignore pip warnings.

For serious projects, users should use conda or docker but this notebook is meant to be very simple and user friendly.

In [ ]:
#Set True if running on NRP Jupyterhub
RUNNING_ON_NRP = False


if (RUNNING_ON_NRP):
    %pip install -e ../. --no-deps

    %pip install xgcm
    %pip install zarr
    %pip install boto3
    %pip install ujson
    %pip install scikit-fmm
    %pip install aiobotocore

# NOTE IF on NRP Jupyterhub, you will likely need to restart the kernel after running this block

## Set AWS credentials
If you are accessing or writing data to S3, you must set credentials.
NOTE: S3 is all that is supported currently.
This typically corresponds to an NRP S3 bucket.

windows:

- `$env:AWS_ACCESS_KEY_ID="..."`
- `$env:AWS_SECRET_ACCESS_KEY="..."`

unix:

- `export AWS_ACCESS_KEY_ID=...`
- `export AWS_SECRET_ACCESS_KEY=...`

# Dataset generation config (quick reference)

This job is fully controlled by `configs/global/run/run.yaml`.  The config
defines **the pipeline variant**, **which dates to process**, and **which
property subsets to compute**.

### Pipeline selection
- `pipeline` in the YAML sets the default (`SURF`, `OSN`, or `DEPTH`).
- `--pipeline` on the CLI always takes precedence.

### Subset selection
- `active_subsets` in the YAML lists which subsets to compute.
- `--subset` on the CLI overrides with a single subset.
- Subset definitions (channel lists, dataset names, depth suffixes) are
  defined in code (`subset_definitions.py`), **not** in the YAML.

### Date iterations
- `data.date_iterations` lists ISO-format date strings.
- For DEPTH: each must match a transferred timestep store in S3.
- For SURF/OSN: each must fall within data availability windows.

### Depth suffixes (DEPTH only)
- Default: `[sfc, z25m, mld, mld_mean]`.
- Override in the YAML with `depth_suffixes: [sfc, mld]` (etc.).

### Output / logging
- `output.bucket`, `output.folder`: S3 location for dataset output.
- `run.run_id`: unique identifier for this session.
- `run.log_dir`: local directory where logs are written.

### Dask note for frontogenesis
The frontogenesis subset merges two large lazy lineages.  The compute function
mitigates this with a single `dask.compute()` call.  If you see scheduler
warnings, reduce `runtime.zarr_async_concurrency` in the config.

# A note about logs

Run logs are stored at `notebooks/notebooks_global/test_logs/{run_id}/`,
resolved relative to the repository root.  Each subset invocation writes
its own log file:

```
test_logs/{run_id}/
    native_fields_20260608_142210.log
    kinematic_20260608_143055.log
    stratification_20260608_150311.log
    run_meta.yaml                       ← full run specification (written once)
```

Each invocation writes its own **timestamped** log file
(`{subset}_{YYYYMMDD_HHMMSS}.log`, run time in UTC), so re-runs never
overwrite or interleave with previous logs.

**Re-running is safe:** the script skips any subset/date whose zarr store
already exists on S3 for this `run_id` (and the exporter skips `.nc`
files that already exist), so a re-run only fills in what is missing.
Pass `--clobber` to force regeneration / re-export.  To start a
completely fresh dataset, use a new `run_id` (it is part of the S3 path).


# Generate a shared run_id for this session

Run this cell **once** at the start of a session.  Reuse `run_id` for every
subset you run below — this groups all output zarr files into the same S3
directory:
```
s3://{bucket}/{folder}/{run_id}/{date_prefix}/kinematic.zarr
s3://{bucket}/{folder}/{run_id}/{date_prefix}/frontogenesis.zarr
... etc.
```

In [2]:
pipeline = "SURF"  # <-- must match configs/global/run/run.yaml

# Output folder is determined by pipeline:
#   SURF / OSN → surface_fields/
#   DEPTH      → depth_fields/
folder = "surface_fields" if pipeline in ("SURF", "OSN") else "depth_fields"

#run_id = f"global_{datetime.datetime.now(datetime.UTC).strftime('%Y%m%d_%H%M%S')}"
run_id = "vfieldtest"  # <-- update if different
print(f"Pipeline: {pipeline}")
print(f"Shared run_id for this session: {run_id}")
print(f"All subsets will write to: s3://dbof/{folder}/{run_id}/")
print()
print("Update configs/global/data_access/access.yaml with this run_id when the run is complete.")

Pipeline: SURF
Shared run_id for this session: vfieldtest
All subsets will write to: s3://dbof/surface_fields/vfieldtest/

Update configs/global/data_access/access.yaml with this run_id when the run is complete.


# Select pipeline variant

Set the pipeline to `SURF`, `OSN`, or `DEPTH`.  This can also be set in
`configs/global/run/run.yaml` (the `pipeline` key) — the CLI `--pipeline` flag
takes precedence.


In [3]:
# Pipeline variant: SURF | OSN | DEPTH
print(f"Pipeline: {pipeline}")

Pipeline: SURF


---
# Run subsets

**IMPORTANT: only run cells that match your pipeline.**
- If `pipeline = SURF` or `OSN` → run cells under **Surface subsets** only.
- If `pipeline = DEPTH` → run cells under **Depth subsets** only.

Running a subset that doesn't belong to the chosen pipeline will raise a
`ValueError`.  See `configs/global/run/run.yaml` for the full mapping.

All cells reuse the `run_id` and `pipeline` set above.
All Dask logs are warnings — do not be alarmed.

### Running multiple subsets in one call

You can also list multiple subsets in `active_subsets` in the config YAML
and omit `--subset` to process them all sequentially.


## Pipeline: SURF / OSN — Surface subsets

**Skip this section if `pipeline = DEPTH`.**

These subsets are available when `pipeline` is `SURF` or `OSN`.


### `native_fields`
Model state variables `Theta`, `Salt`, `Eta`, `W`, plus velocity `U`, `V` interpolated to tracer points and rotated to eastward/northward.

In [6]:
!generate-global \
    --config ../../configs/global/run/run_field_consolidation_test.yaml \
    --pipeline $pipeline \
    --subset native_fields \
    --run_id $run_id

2026-07-31 08:50:53,023 | INFO | Log file: /home/lhoffma2/git/llc4320-native-grid-preprocessing/notebooks/notebooks_global/test_logs/vfieldtest/native_fields_20260731_155053.log
2026-07-31 08:50:53,023 | INFO | Unified global pipeline starting.
2026-07-31 08:50:53,023 | INFO | Pipeline: SURF
2026-07-31 08:50:53,023 | INFO | Active subsets: ['native_fields']
2026-07-31 08:50:53,023 | INFO | Depth suffixes (YAML override): ['sfc', 'mld']
2026-07-31 08:50:53,023 | INFO | Dates: ['2012-05-01 12:00:00']
2026-07-31 08:50:53,024 | INFO | Pre-flight plan (zarr existence per subset/date):
2026-07-31 08:50:53,032 | INFO | Found credentials in shared credentials file: ~/.aws/credentials
2026-07-31 08:50:53,135 | INFO |   native_fields          2012-05-01 12:00:00  ->  GENERATE (zarr store missing)
2026-07-31 08:50:53,562 | INFO | State start
2026-07-31 08:50:53,565 | INFO |   Scheduler at:     tcp://127.0.0.1:41213
2026-07-31 08:50:53,566 | INFO |   dashboard at:  http://127.0.0.1:8787/status
202

### `frontal_structure`
`gradb2`, `gradsalt2`, `gradtheta2`, `gradeta2`, `gradrho2`, `turner_angle`, `density`, `buoyancy`.

In [4]:
!generate-global \
    --config ../../configs/global/run/run_field_consolidation_test.yaml \
    --pipeline $pipeline \
    --subset frontal_structure \
    --run_id $run_id

2026-07-31 08:43:25,604 | INFO | Log file: /home/lhoffma2/git/llc4320-native-grid-preprocessing/notebooks/notebooks_global/test_logs/vfieldtest/frontal_structure_20260731_154325.log
2026-07-31 08:43:25,604 | INFO | Unified global pipeline starting.
2026-07-31 08:43:25,604 | INFO | Pipeline: SURF
2026-07-31 08:43:25,604 | INFO | Active subsets: ['frontal_structure']
2026-07-31 08:43:25,604 | INFO | Depth suffixes (YAML override): ['sfc', 'mld']
2026-07-31 08:43:25,604 | INFO | Dates: ['2012-05-01 12:00:00']
2026-07-31 08:43:25,605 | INFO | Pre-flight plan (zarr existence per subset/date):
2026-07-31 08:43:25,613 | INFO | Found credentials in shared credentials file: ~/.aws/credentials
2026-07-31 08:43:26,353 | INFO |   frontal_structure      2012-05-01 12:00:00  ->  GENERATE (zarr store missing)
2026-07-31 08:43:26,899 | INFO | State start
2026-07-31 08:43:26,903 | INFO |   Scheduler at:     tcp://127.0.0.1:33325
2026-07-31 08:43:26,903 | INFO |   dashboard at:  http://127.0.0.1:8787/st

### `kinematic`
Velocity-derived scalar fields from a single Jacobian pass.

In [9]:
!generate-global \
    --config ../../configs/global/run/run_field_consolidation_test.yaml \
    --pipeline $pipeline \
    --subset kinematic \
    --run_id $run_id

2026-07-31 09:11:16,045 | INFO | Log file: /home/lhoffma2/git/llc4320-native-grid-preprocessing/notebooks/notebooks_global/test_logs/vfieldtest/kinematic_20260731_161116.log
2026-07-31 09:11:16,045 | INFO | Unified global pipeline starting.
2026-07-31 09:11:16,046 | INFO | Pipeline: SURF
2026-07-31 09:11:16,046 | INFO | Active subsets: ['kinematic']
2026-07-31 09:11:16,046 | INFO | Depth suffixes (YAML override): ['sfc', 'mld']
2026-07-31 09:11:16,046 | INFO | Dates: ['2012-05-01 12:00:00']
2026-07-31 09:11:16,049 | INFO | Pre-flight plan (zarr existence per subset/date):
2026-07-31 09:11:16,057 | INFO | Found credentials in shared credentials file: ~/.aws/credentials
2026-07-31 09:11:16,198 | INFO |   kinematic              2012-05-01 12:00:00  ->  GENERATE (zarr store missing)
2026-07-31 09:11:16,648 | INFO | State start
2026-07-31 09:11:16,653 | INFO |   Scheduler at:     tcp://127.0.0.1:41687
2026-07-31 09:11:16,653 | INFO |   dashboard at:  http://127.0.0.1:8787/status
2026-07-31 

### `frontogenesis`
`frontogenesis_tendency`, `ug`, `vg`, `frontogenesis_geo`, `frontogenesis_ageo`.

In [10]:
!generate-global \
    --config ../../configs/global/run/run_field_consolidation_test.yaml \
    --pipeline $pipeline \
    --subset frontogenesis \
    --run_id $run_id

2026-07-31 09:21:22,022 | INFO | Log file: /home/lhoffma2/git/llc4320-native-grid-preprocessing/notebooks/notebooks_global/test_logs/vfieldtest/frontogenesis_20260731_162122.log
2026-07-31 09:21:22,022 | INFO | Unified global pipeline starting.
2026-07-31 09:21:22,022 | INFO | Pipeline: SURF
2026-07-31 09:21:22,022 | INFO | Active subsets: ['frontogenesis']
2026-07-31 09:21:22,022 | INFO | Depth suffixes (YAML override): ['sfc', 'mld']
2026-07-31 09:21:22,022 | INFO | Dates: ['2012-05-01 12:00:00']
2026-07-31 09:21:22,023 | INFO | Pre-flight plan (zarr existence per subset/date):
2026-07-31 09:21:22,031 | INFO | Found credentials in shared credentials file: ~/.aws/credentials
2026-07-31 09:21:22,166 | INFO |   frontogenesis          2012-05-01 12:00:00  ->  GENERATE (zarr store missing)
2026-07-31 09:21:22,656 | INFO | State start
2026-07-31 09:21:22,660 | INFO |   Scheduler at:     tcp://127.0.0.1:33809
2026-07-31 09:21:22,661 | INFO |   dashboard at:  http://127.0.0.1:8787/status
202

### `icearea`
sea ice area

In [8]:
!generate-global \
    --config ../../configs/global/run/run_field_consolidation_test.yaml \
    --pipeline $pipeline \
    --subset icearea \
    --run_id $run_id

2026-07-31 09:08:54,005 | INFO | Log file: /home/lhoffma2/git/llc4320-native-grid-preprocessing/notebooks/notebooks_global/test_logs/vfieldtest/icearea_20260731_160854.log
2026-07-31 09:08:54,005 | INFO | Unified global pipeline starting.
2026-07-31 09:08:54,005 | INFO | Pipeline: SURF
2026-07-31 09:08:54,005 | INFO | Active subsets: ['icearea']
2026-07-31 09:08:54,005 | INFO | Depth suffixes (YAML override): ['sfc', 'mld']
2026-07-31 09:08:54,005 | INFO | Dates: ['2012-05-01 12:00:00']
2026-07-31 09:08:54,006 | INFO | Pre-flight plan (zarr existence per subset/date):
2026-07-31 09:08:54,018 | INFO | Found credentials in shared credentials file: ~/.aws/credentials
2026-07-31 09:08:54,119 | INFO |   icearea                2012-05-01 12:00:00  ->  GENERATE (zarr store missing)
2026-07-31 09:08:54,629 | INFO | State start
2026-07-31 09:08:54,633 | INFO |   Scheduler at:     tcp://127.0.0.1:34253
2026-07-31 09:08:54,634 | INFO |   dashboard at:  http://127.0.0.1:8787/status
2026-07-31 09:0

### `surface winds`


In [7]:
!generate-global \
    --config ../../configs/global/run/run_field_consolidation_test.yaml \
    --pipeline $pipeline \
    --subset surface_wind \
    --run_id $run_id

2026-07-31 08:58:39,678 | INFO | Log file: /home/lhoffma2/git/llc4320-native-grid-preprocessing/notebooks/notebooks_global/test_logs/vfieldtest/surface_wind_20260731_155839.log
2026-07-31 08:58:39,678 | INFO | Unified global pipeline starting.
2026-07-31 08:58:39,678 | INFO | Pipeline: SURF
2026-07-31 08:58:39,678 | INFO | Active subsets: ['surface_wind']
2026-07-31 08:58:39,678 | INFO | Depth suffixes (YAML override): ['sfc', 'mld']
2026-07-31 08:58:39,678 | INFO | Dates: ['2012-05-01 12:00:00']
2026-07-31 08:58:39,680 | INFO | Pre-flight plan (zarr existence per subset/date):
2026-07-31 08:58:39,688 | INFO | Found credentials in shared credentials file: ~/.aws/credentials
2026-07-31 08:58:39,974 | INFO |   surface_wind           2012-05-01 12:00:00  ->  GENERATE (zarr store missing)
2026-07-31 08:58:40,454 | INFO | State start
2026-07-31 08:58:40,457 | INFO |   Scheduler at:     tcp://127.0.0.1:38201
2026-07-31 08:58:40,457 | INFO |   dashboard at:  http://127.0.0.1:8787/status
2026-

## Pipeline: DEPTH — Depth subsets

**Skip this section if `pipeline = SURF` or `OSN`.**

These subsets are available when `pipeline` is `DEPTH`.  All compute from
full-depth 3D fields and reduce to 2D surface output via depth strategies.


### `stratification`
MLD, N² at 4 depths, ML heat content.

In [4]:
!generate-global \
    --config ../../configs/global/run/run.yaml \
    --pipeline $pipeline \
    --subset stratification \
    --run_id $run_id

2026-06-05 22:31:50,010 | INFO | Log file: /home/lhoffma2/git/llc4320-native-grid-preprocessing/notebooks/notebooks_global/test_logs/global_DEPTH_test01/stratification.log
2026-06-05 22:31:50,010 | INFO | Unified global pipeline starting.
2026-06-05 22:31:50,010 | INFO | Pipeline: DEPTH
2026-06-05 22:31:50,011 | INFO | Active subsets: ['stratification']
2026-06-05 22:31:50,011 | INFO | Depth suffixes (YAML override): ['sfc', 'mld']
2026-06-05 22:31:50,011 | INFO | Dates: ['2012-11-09 12:00:00']
2026-06-05 22:31:50,380 | INFO | To route to workers diagnostics web server please install jupyter-server-proxy: python -m pip install jupyter-server-proxy
2026-06-05 22:31:50,394 | INFO | State start
2026-06-05 22:31:50,397 | INFO |   Scheduler at:     tcp://127.0.0.1:41953
2026-06-05 22:31:50,398 | INFO |   dashboard at:  http://127.0.0.1:8787/status
2026-06-05 22:31:50,398 | INFO | Registering Worker plugin shuffle
2026-06-05 22:31:50,407 | INFO |         Start Nanny at: 'tcp://127.0.0.1:4586

### `vertical_shear`
Vertical shear and Richardson number at 4 depths.

In [ ]:
!generate-global \
    --config ../../configs/global/run/run.yaml \
    --pipeline $pipeline \
    --subset vertical_shear \
    --run_id $run_id

### `mixing_parameters`
Fr, Ro, Bu at 4 depths.

In [ ]:
!generate-global \
    --config ../../configs/global/run/run.yaml \
    --pipeline $pipeline \
    --subset mixing_parameters \
    --run_id $run_id

### `ertel_pv`
Ertel PV and its vertical/tilt components at 4 depths (12 channels).

In [ ]:
!generate-global \
    --config ../../configs/global/run/run.yaml \
    --pipeline $pipeline \
    --subset ertel_pv \
    --run_id $run_id

### `buoyancy_fluxes`
uB, vB, wB at 4 depths (12 channels).

In [ ]:
!generate-global \
    --config ../../configs/global/run/run.yaml \
    --pipeline $pipeline \
    --subset buoyancy_fluxes \
    --run_id $run_id

### `surface_wind`
Geographic wind stress (`oceTAUX` eastward, `oceTAUY` northward), `wind_stress_curl`, Ekman pumping/transport, plus `oceQnet` (SURF/DEPTH only).

In [ ]:
!generate-global \
    --config ../../configs/global/run/run.yaml \
    --pipeline $pipeline \
    --subset surface_wind \
    --run_id $run_id

### `energetics`
Kinetic energy at 4 depths.

In [ ]:
!generate-global \
    --config ../../configs/global/run/run.yaml \
    --pipeline $pipeline \
    --subset energetics \
    --run_id $run_id

### `icearea`
Sea-ice area fraction.

In [ ]:
!generate-global \
    --config ../../configs/global/run/run.yaml \
    --pipeline $pipeline \
    --subset icearea \
    --run_id $run_id

---
# S3 management

In [ ]:
# aws cli commands for listing or deleting data in the s3 bucket
# Replace {run_id} with your run_id value, or use the f-string version below.

# List all zarr stores for this run:
# aws --endpoint https://s3-west.nrp-nautilus.io s3 ls s3://dbof/{folder}/{run_id}/ --human-readable

# Delete an individual subset zarr (dry-run first):
# aws --endpoint https://s3-west.nrp-nautilus.io s3 rm s3://dbof/{folder}/{run_id}/20121109_120000/kinematic.zarr --recursive --dryrun

# Delete the entire run directory (dry-run first):
# aws --endpoint https://s3-west.nrp-nautilus.io s3 rm s3://dbof/{folder}/{run_id}/ --recursive --dryrun